# nb01_data_quality
**Chủ:** A  
Chứng minh các lỗi dữ liệu bằng số tự đếm (mục 2 của kế hoạch)


## 0. Nạp dữ liệu và kiểm tra kích thước
Đọc hai file CSV từ `data/raw/`. Kích thước phải khớp số đã đo ở A1: app 10 841 × 13, review 64 295 × 5.

In [3]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
app = pd.read_csv(RAW / "googleplaystore.csv")
rev = pd.read_csv(RAW / "googleplaystore_user_reviews.csv")
print("app:", app.shape, "| rev:", rev.shape)

app: (10841, 13) | rev: (64295, 5)


## Vấn đề 1: Dòng lệch cột
Một dòng bị thiếu trường nên mọi giá trị dịch trái một cột. Dấu hiệu: `Rating` vượt miền hợp lệ 1–5.
Bằng chứng: số dòng có Rating > 5, và nội dung dòng đó.

In [4]:
rating_num = pd.to_numeric(app["Rating"], errors="coerce")
mis = app[rating_num > 5]
print("Rating > 5:", len(mis), "| max:", rating_num.max())
mis

Rating > 5: 1 | max: 19.0


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,"February 11, 2018",1.0.19,4.0 and up,NaN


## Vấn đề 2: Dòng trùng hoàn toàn (bảng app)
Đếm các dòng giống nhau ở mọi cột, sau khi đã bỏ dòng lệch cột (A0).

In [5]:
a0 = app[~(rating_num > 5)]
print("a0:", len(a0))
print("trùng hoàn toàn:", a0.duplicated().sum())

a0: 10840
trùng hoàn toàn: 483


## Vấn đề 3: Cùng tên app, khác thuộc tính
Sau khi bỏ trùng hoàn toàn (A1), so số dòng với số tên app duy nhất. Chênh lệch là các app bị lặp với thuộc tính khác nhau.
Hệ quả: khoá nối review → app chỉ là tên app, nên tên lặp làm mất tính duy nhất.

In [6]:
a1 = a0.drop_duplicates()
print("a1:", len(a1), "| tên duy nhất:", a1["App"].nunique())
print("dòng thừa theo tên:", len(a1) - a1["App"].nunique())
a1[a1["App"].duplicated(keep=False)].sort_values("App").head(10)

a1: 10357 | tên duy nhất: 9659
dòng thừa theo tên: 698


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
5415,365Scores - Live Scores,SPORTS,4.6,666246,25M,"10,000,000+",Free,0,Everyone,Sports,"July 29, 2018",5.5.9,4.1 and up
3083,365Scores - Live Scores,SPORTS,4.6,666521,25M,"10,000,000+",Free,0,Everyone,Sports,"July 29, 2018",5.5.9,4.1 and up
1755,8 Ball Pool,GAME,4.5,14200344,52M,"100,000,000+",Free,0,Everyone,Sports,"July 31, 2018",4.0.0,4.0.3 and up
1703,8 Ball Pool,GAME,4.5,14198602,52M,"100,000,000+",Free,0,Everyone,Sports,"July 31, 2018",4.0.0,4.0.3 and up
1871,8 Ball Pool,GAME,4.5,14201891,52M,"100,000,000+",Free,0,Everyone,Sports,"July 31, 2018",4.0.0,4.0.3 and up
1844,8 Ball Pool,GAME,4.5,14200550,52M,"100,000,000+",Free,0,Everyone,Sports,"July 31, 2018",4.0.0,4.0.3 and up
1675,8 Ball Pool,GAME,4.5,14198297,52M,"100,000,000+",Free,0,Everyone,Sports,"July 31, 2018",4.0.0,4.0.3 and up
1970,8 Ball Pool,GAME,4.5,14201604,52M,"100,000,000+",Free,0,Everyone,Sports,"July 31, 2018",4.0.0,4.0.3 and up
3953,8 Ball Pool,SPORTS,4.5,14184910,52M,"100,000,000+",Free,0,Everyone,Sports,"July 31, 2018",4.0.0,4.0.3 and up
559,95Live -SG#1 Live Streaming App,DATING,4.1,4953,15M,"1,000,000+",Free,0,Teen,Dating,"August 1, 2018",8.7.2,4.2 and up


## Vấn đề 4: Rating thiếu
Đếm số app không có điểm và tỉ lệ trên A1. Không điền giá trị; chỉ ghi nhận để nêu giới hạn kết luận.

In [7]:
n_miss = a1["Rating"].isna().sum()
print("Rating thiếu:", n_miss, f"({n_miss/len(a1):.1%})")

Rating thiếu: 1465 (14.1%)


## Vấn đề 5: Cột Size trộn đơn vị
`Size` có đuôi M, k và giá trị chữ "Varies with device". Đếm số dòng "Varies" và xem phân bố đuôi đơn vị.

In [8]:
n_var = (a1["Size"] == "Varies with device").sum()
print("Size 'Varies with device':", n_var, f"({n_var/len(a1):.1%})")
print(a1["Size"].str[-1].value_counts().head())   # đuôi M / k / khác

Size 'Varies with device': 1526 (14.7%)
Size
M    8516
e    1526
k     315
Name: count, dtype: int64


## Vấn đề 6: Installs và Price là chuỗi có ký tự
`Installs` có dấu `+` và dấu phẩy, `Price` có ký hiệu `$`. Không tính toán trực tiếp được cho đến khi đổi sang số.

In [9]:
print("Installs khác nhau:", a1["Installs"].nunique())
print(a1["Installs"].value_counts().head(5))
print("Price có '$':", a1["Price"].str.contains(r"\$", na=False).sum())

Installs khác nhau: 21
Installs
1,000,000+     1488
10,000,000+    1132
100,000+       1129
10,000+        1033
1,000+          890
Name: count, dtype: int64
Price có '$': 765


## Vấn đề 7: Review trùng và review trống (bảng review)
Đếm dòng trùng hoàn toàn và dòng không có văn bản (`Translated_Review` rỗng). Đo trên bảng review thô, độc lập với bảng app.

In [10]:
n_dup = rev.duplicated().sum()
n_empty = rev["Translated_Review"].isna().sum()
print("review trùng:", n_dup, f"({n_dup/len(rev):.1%})")
print("review trống:", n_empty, f"({n_empty/len(rev):.1%})")

review trùng: 33616 (52.3%)
review trống: 26868 (41.8%)
